In [ ]:
!pip install opencv-python pydicom numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 16.6 MB/s eta 0:00:00


In [2]:
!wget -c https://s3.amazonaws.com/fast-ai-imagelocal/siim_small.tgz
!tar -xzf siim_small.tgz

--2025-05-12 18:07:33--  https://s3.amazonaws.com/fast-ai-imagelocal/siim_small.tgz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.229.176, 54.231.129.104, 52.216.53.160, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.229.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 33276453 (32M) [application/x-tar]
Saving to: ‘siim_small.tgz’

siim_small.tgz      100%[===================>]  31.73M   103MB/s    in 0.3s    

2025-05-12 18:07:33 (103 MB/s) - ‘siim_small.tgz’ saved [33276453/33276453]



In [25]:
import cv2
import pydicom
import numpy as np
import os

# 1. Função para carregar e pré-processar imagens
def carregar_imagens(arquivo_txt, classe):
    with open(arquivo_txt, 'r') as f:
        caminhos = [linha.strip() for linha in f.readlines()] # Lê os caminhos das imagens

    imagens = []
    for caminho in caminhos[:15]:  # Limita a 15 imagens por classe
        try:
            # Carrega a imagem DICOM
            ds = pydicom.dcmread(os.path.join('siim_small', caminho))
            img = ds.pixel_array # Extrai a imagem como array

            # Normalização e equalização do histograma
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
            img = cv2.equalizeHist(img)

            imagens.append(img)

        except Exception as e:
            print(f"Erro ao processar {caminho}: {e}")

    print(f"Carregadas {len(imagens)} imagens de {classe}")
    return imagens

# 2. Carrega as imagens
print("Carregando imagens...")
pneumo_imgs = carregar_imagens('Pneumothorax_files.txt', 'Pneumothorax')
no_pneumo_imgs = carregar_imagens('No_Pneumothorax_files.txt', 'No Pneumothorax')

# 3. Função para classificação por histograma
def classificar_histograma(img_teste, base_imgs, base_rotulos, metodo):
    hist_teste = cv2.calcHist([img_teste], [0], None, [256], [0, 256])

    # Inicializa o melhor score dependendo do método
    melhor_score = -1 if metodo in [cv2.HISTCMP_CORREL, cv2.HISTCMP_INTERSECT] else float('inf')
    melhor_rotulo = None

    # Compara com todas as imagens da base
    for img, rotulo in zip(base_imgs, base_rotulos):
        hist = cv2.calcHist([img], [0], None, [256], [0, 256])
        score = cv2.compareHist(hist_teste, hist, metodo)

        # Atualiza o melhor score e rótulo conforme o critério do método
        if (metodo in [cv2.HISTCMP_CORREL, cv2.HISTCMP_INTERSECT] and score > melhor_score) or \
           (metodo not in [cv2.HISTCMP_CORREL, cv2.HISTCMP_INTERSECT] and score < melhor_score):
            melhor_score = score
            melhor_rotulo = rotulo

    return melhor_rotulo

# 4. Métodos de comparação
metodos = {
    "CORREL": cv2.HISTCMP_CORREL, # Correlação
    "CHISQR": cv2.HISTCMP_CHISQR, # Qui-quadrado
    "INTERSECT": cv2.HISTCMP_INTERSECT, # Interseção
    "BHATTACHARYYA": cv2.HISTCMP_BHATTACHARYYA # Distância de Bhattacharyya
}

# 5. Avaliação
print("\nAvaliando métodos...")
with open("resultados.txt", "w") as f:
    for nome, metodo in metodos.items():
        print(f"Método: {nome}")
        f.write(f"=== Método: {nome} ===\n")

        # Listas para métricas
        classe_real = []
        classe_indicada = []

        # Todas as imagens
        todas_imagens = pneumo_imgs + no_pneumo_imgs
        todas_rotulos = [1]*len(pneumo_imgs) + [0]*len(no_pneumo_imgs) # 1 para pneumo e 0 para não pneumo

        for i in range(len(todas_imagens)):
            img_teste = todas_imagens[i]
            rotulo_teste = todas_rotulos[i]

            # Define conjunto de treino (todas menos a atual)
            img_treino = [img for j, img in enumerate(todas_imagens) if j != i]
            rotulo_treino = [rot for j, rot in enumerate(todas_rotulos) if j != i]


            indicada = classificar_histograma(img_teste, img_treino, rotulo_treino, metodo)

            # Salva a classe real e a indicada
            classe_real.append(rotulo_teste)
            classe_indicada.append(indicada)

        # Cálculo das métricas
        VP = sum((t == 1 and p== 1) for t, p in zip(classe_real, classe_indicada)) # Verdadeiros Positivos
        VN = sum((t == 0 and p == 0) for t, p in zip(classe_real, classe_indicada)) # Verdadeiros Negativos
        FP = sum((t == 0 and p == 1) for t, p in zip(classe_real, classe_indicada)) # Falsos Positivos
        FN = sum((t == 1 and p == 0) for t, p in zip(classe_real, classe_indicada)) # Falsos negativos

        # Matriz de confusão
        matriz_confusao = f"""
        Matriz de Confusão:
        ------------------
        | VP: {VP} | FP: {FP} |
        | FN: {FN} | VN: {VN} |
        ------------------
        """

        # Cálculo das métricas
        sensibilidade = VP / (VP + FN) if (VP + FN) > 0 else 0
        especificidade = VN / (VN + FP) if (VN + FP) > 0 else 0

        # Escreve os resultados no arquivo
        f.write(matriz_confusao)
        f.write("\n")
        f.write(f"Sensibilidade: {sensibilidade:.2f}\n")
        f.write(f"Especificidade: {especificidade:.2f}\n")
        f.write("\n")


print("\nProcesso concluído! Resultados salvos em resultados.txt")

Carregando imagens...
Carregadas 15 imagens de Pneumothorax
Carregadas 15 imagens de No Pneumothorax

Avaliando métodos...
Método: CORREL
Método: CHISQR
Método: INTERSECT
Método: BHATTACHARYYA

Processo concluído! Resultados salvos em resultados.txt
